# Timestomping Detection Tool - Run Detection

This notebook loads the trained XGBoost model (48 hybrid features) and generates predictions on the feature dataset.

## Input

- `data_features.csv` from notebook 02 (48 features)
- `best_model_hybrid_48features.pkl` from Phase 3 training
- `feature_columns_hybrid_48features.pkl` for feature names

## Process

1. Load trained model and feature columns
2. Prepare features (48 hybrid features)
3. Convert timestamp strings to Unix timestamps
4. Generate predictions with confidence scores
5. Validate against ground truth (if available)
6. Analyze feature importance
7. Save outputs

## Output

- `predictions.csv` - All records with confidence scores
- `predictions_with_features.csv` - Predictions with feature values for analysis
- `flagged_files.csv` - High-confidence detections only


## Cell 1: Imports and Setup

In [452]:
# Cell 1: Imports and Setup

import pandas as pd
import numpy as np
import os
import pickle
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")


Libraries imported successfully
Pandas version: 2.3.2
NumPy version: 2.3.3


## Cell 2: User Configuration

**EDIT THIS SECTION** if you changed paths in previous notebooks


In [453]:
# Cell 2: User Configuration

# Input file (output from notebook 02)
INPUT_DIR = '/Users/soni/Github/Digital-Detectives_Thesis/data/processed/Prototype Tool Output/LW/XGBoostHybrid'
INPUT_FILE = "data_features.csv"

# Model files (48-feature XGBoost model from Phase 3 training)
MODEL_PATH = "/Users/soni/Github/Digital-Detectives_Thesis/models/for autopsy/best_model_hybrid_48features.pkl"
FEATURE_COLS_PATH = "/Users/soni/Github/Digital-Detectives_Thesis/models/for autopsy/feature_columns_hybrid_48features.pkl"

# Output directory
OUTPUT_DIR = INPUT_DIR

# Detection threshold (confidence percentage)
CONFIDENCE_THRESHOLD = 80  # Flag files with ≥70% confidence

# Construct paths
input_path = os.path.join(INPUT_DIR, INPUT_FILE)
output_predictions = os.path.join(OUTPUT_DIR, "predictions.csv")
output_predictions_features = os.path.join(OUTPUT_DIR, "predictions_with_features.csv")
output_flagged = os.path.join(OUTPUT_DIR, "flagged_files.csv")

print("Configuration loaded")
print("-" * 80)
print(f"Input file: {input_path}")
print(f"  Exists: {os.path.exists(input_path)}")
print(f"Model file: {MODEL_PATH}")
print(f"  Exists: {os.path.exists(MODEL_PATH)}")
print(f"Feature columns file: {FEATURE_COLS_PATH}")
print(f"  Exists: {os.path.exists(FEATURE_COLS_PATH)}")
print(f"Confidence threshold: {CONFIDENCE_THRESHOLD}%")
print("-" * 80)
print(f"Output files:")
print(f"  - {output_predictions}")
print(f"  - {output_predictions_features}")
print(f"  - {output_flagged}")

if not os.path.exists(input_path):
    raise FileNotFoundError(f"Feature data not found: {input_path}")
if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(f"Model not found: {MODEL_PATH}")
if not os.path.exists(FEATURE_COLS_PATH):
    raise FileNotFoundError(f"Feature columns file not found: {FEATURE_COLS_PATH}")


Configuration loaded
--------------------------------------------------------------------------------
Input file: /Users/soni/Github/Digital-Detectives_Thesis/data/processed/Prototype Tool Output/LW/XGBoostHybrid/data_features.csv
  Exists: True
Model file: /Users/soni/Github/Digital-Detectives_Thesis/models/for autopsy/best_model_hybrid_48features.pkl
  Exists: True
Feature columns file: /Users/soni/Github/Digital-Detectives_Thesis/models/for autopsy/feature_columns_hybrid_48features.pkl
  Exists: True
Confidence threshold: 80%
--------------------------------------------------------------------------------
Output files:
  - /Users/soni/Github/Digital-Detectives_Thesis/data/processed/Prototype Tool Output/LW/XGBoostHybrid/predictions.csv
  - /Users/soni/Github/Digital-Detectives_Thesis/data/processed/Prototype Tool Output/LW/XGBoostHybrid/predictions_with_features.csv
  - /Users/soni/Github/Digital-Detectives_Thesis/data/processed/Prototype Tool Output/LW/XGBoostHybrid/flagged_files.c

## Cell 3: Load Feature Dataset


In [454]:
# Cell 3: Load Feature Dataset

print("\n" + "="*80)
print("LOADING FEATURE DATASET")
print("="*80)

df = pd.read_csv(input_path, low_memory=False)

print(f"\nDataset loaded successfully")
print(f"  Records: {len(df):,}")
print(f"  Columns: {len(df.columns)}")
print(f"  Memory: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

# Check if ground truth available
if 'ground_truth_label' in df.columns:
    print(f"\nGround truth available:")
    print(f"  Suspicious: {df['ground_truth_label'].sum():,}")
    print(f"  Benign: {(df['ground_truth_label'] == 0).sum():,}")
else:
    print(f"\nNo ground truth (production mode)")



LOADING FEATURE DATASET

Dataset loaded successfully
  Records: 7,420
  Columns: 83
  Memory: 12.23 MB

Ground truth available:
  Suspicious: 0
  Benign: 7,420


## Cell 4: Load Trained Model and Feature Columns


In [455]:
# Cell 4: Load Trained Model and Feature Columns

print("\n" + "=" * 80)
print("LOADING TRAINED MODEL AND FEATURE COLUMNS")
print("=" * 80)

# Load model
with open(MODEL_PATH, 'rb') as f:
    model = pickle.load(f)

print(f"\nModel loaded successfully")
print(f"  Model type: {type(model).__name__}")

# Load feature columns
with open(FEATURE_COLS_PATH, 'rb') as f:
    feature_cols = pickle.load(f)

print(f"\nFeature columns loaded")
print(f"  Expected features: {len(feature_cols)}")
print(f"\nFirst 10 features:")
for i, feat in enumerate(feature_cols[:10], 1):
    print(f"  {i:2d}. {feat}")
print(f"  ...")



LOADING TRAINED MODEL AND FEATURE COLUMNS

Model loaded successfully
  Model type: XGBClassifier

Feature columns loaded
  Expected features: 48

First 10 features:
   1. zero_in_nanoseconds_lf
   2. zero_in_nanoseconds_suspicious
   3. zero_in_nanoseconds
   4. time_reversal_event
   5. basic_info_changed
   6. using_another_timestamp
   7. si_timestamp_changed
   8. update_resident_value
   9. creation_time_modified
  10. modified_time_modified
  ...


## Cell 5: Prepare Features for Prediction

Extract 48 hybrid features and convert timestamp strings to Unix timestamps


In [456]:
# Cell 5: Prepare Features for Prediction

print("\n" + "=" * 80)
print("PREPARING FEATURES FOR PREDICTION")
print("=" * 80)

print(f"\nExtracting {len(feature_cols)} features...")

# Select features in correct order
X = df[feature_cols].copy()

# Convert timestamp strings to Unix timestamps (seconds since epoch)
timestamp_cols = [
    'lf_creation_time_before', 'lf_creation_time_after',
    'lf_modified_time_before', 'lf_modified_time_after',
    'lf_accessed_time_before', 'lf_accessed_time_after'
]

for col in timestamp_cols:
    if col in X.columns:
        print(f"  Converting {col} to Unix timestamp...")
        X[col] = pd.to_datetime(X[col], errors='coerce')
        X[col] = X[col].astype('int64') / 10**9  # Convert to seconds
        X[col] = X[col].fillna(0)

# Handle any remaining missing values
X = X.fillna(0)

# Convert boolean to int
for col in X.columns:
    if X[col].dtype == 'bool':
        X[col] = X[col].astype(int)

print(f"\nFeature matrix prepared:")
print(f"  Shape: {X.shape}")
print(f"  Features: {len(X.columns)}")
print(f"  Records: {len(X):,}")



PREPARING FEATURES FOR PREDICTION

Extracting 48 features...
  Converting lf_creation_time_before to Unix timestamp...
  Converting lf_creation_time_after to Unix timestamp...
  Converting lf_modified_time_before to Unix timestamp...
  Converting lf_modified_time_after to Unix timestamp...
  Converting lf_accessed_time_before to Unix timestamp...
  Converting lf_accessed_time_after to Unix timestamp...

Feature matrix prepared:
  Shape: (7420, 48)
  Features: 48
  Records: 7,420


In [457]:
# Cell 6: Generate Predictions

print("\n" + "=" * 80)
print("GENERATING PREDICTIONS")
print("=" * 80)

# Generate probability predictions
print("\nRunning model predictions...")
y_pred_proba = model.predict_proba(X)

# Extract probability for positive class (timestomped)
if y_pred_proba.ndim == 2:
    # Binary classification - take probability of class 1
    prediction_proba = y_pred_proba[:, 1]
else:
    # Single probability output
    prediction_proba = y_pred_proba

# Convert to confidence percentage
confidence_pct = prediction_proba * 100

# Binary predictions (threshold at confidence_threshold)
prediction = (confidence_pct >= CONFIDENCE_THRESHOLD).astype(int)

print(f"\nEvent-level predictions:")
print(f"  Total events: {len(df):,}")
print(f"  Flagged events (≥{CONFIDENCE_THRESHOLD}% confidence): {prediction.sum():,}")
print(f"  Benign events (<{CONFIDENCE_THRESHOLD}% confidence): {(prediction == 0).sum():,}")

# Add predictions to dataframe
df['prediction'] = prediction
df['confidence_pct'] = confidence_pct

# File-level aggregation (max confidence per file)
print(f"\nFile-level aggregation:")
file_level = df.groupby('filename').agg({
    'prediction': 'max',
    'confidence_pct': 'max'
}).reset_index()

flagged_files = file_level[file_level['prediction'] == 1]
print(f"  Unique files: {len(file_level):,}")
print(f"  Flagged files: {len(flagged_files):,}")
print(f"  Detection rate: {(len(flagged_files) / len(file_level) * 100):.2f}%")

# Display confidence distribution
print(f"\nConfidence distribution (event-level):")
print(f"  Minimum: {confidence_pct.min():.2f}%")
print(f"  Maximum: {confidence_pct.max():.2f}%")
print(f"  Mean: {confidence_pct.mean():.2f}%")
print(f"  Median: {np.median(confidence_pct):.2f}%")



GENERATING PREDICTIONS

Running model predictions...

Event-level predictions:
  Total events: 7,420
  Flagged events (≥80% confidence): 24
  Benign events (<80% confidence): 7,396

File-level aggregation:
  Unique files: 7,415
  Flagged files: 24
  Detection rate: 0.32%

Confidence distribution (event-level):
  Minimum: 0.00%
  Maximum: 99.95%
  Mean: 1.11%
  Median: 0.00%


## Cell 7: Validation Against Ground Truth (if available)

Validate predictions against ground truth labels at both event and file level


In [458]:
# Cell 7: Validation Against Ground Truth (if available)

print("\n" + "=" * 80)
print("VALIDATION AGAINST GROUND TRUTH")
print("=" * 80)

if 'ground_truth_label' in df.columns:
    print("\nGround truth found - validating predictions...")
    
    # Event-level metrics
    from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix
    
    y_true = df['ground_truth_label']
    y_pred = df['prediction']
    
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    
    print(f"\nEvent-level metrics:")
    print(f"  Precision: {precision:.2%}")
    print(f"  Recall: {recall:.2%}")
    print(f"  F1-Score: {f1:.4f}")
    print(f"\nConfusion Matrix:")
    print(f"  True Positives (TP): {tp:,}")
    print(f"  False Positives (FP): {fp:,}")
    print(f"  True Negatives (TN): {tn:,}")
    print(f"  False Negatives (FN): {fn:,}")
    
    # File-level validation
    print(f"\n" + "-" * 80)
    print("FILE-LEVEL VALIDATION")
    print("-" * 80)
    
    # Get ground truth files
    ground_truth_files = df[df['ground_truth_label'] == 1]['filename'].unique()
    detected_files = df[df['prediction'] == 1]['filename'].unique()
    
    # Calculate file-level metrics
    correctly_detected = set(ground_truth_files) & set(detected_files)
    missed_files = set(ground_truth_files) - set(detected_files)
    false_positive_files = set(detected_files) - set(ground_truth_files)
    
    file_precision = len(correctly_detected) / len(detected_files) if len(detected_files) > 0 else 0
    file_recall = len(correctly_detected) / len(ground_truth_files) if len(ground_truth_files) > 0 else 0
    file_f1 = 2 * (file_precision * file_recall) / (file_precision + file_recall) if (file_precision + file_recall) > 0 else 0
    
    print(f"\nGround truth: {len(ground_truth_files)} timestomped files")
    print(f"Model flagged: {len(detected_files)} files")
    print(f"\nDetection results:")
    print(f"  Detected: {len(correctly_detected)}/{len(ground_truth_files)} ({file_recall:.1%})")
    print(f"  Missed: {len(missed_files)}/{len(ground_truth_files)}")
    
    if missed_files:
        print(f"\nMissed files:")
        for f in sorted(missed_files):
            print(f"  - {f}")
    
    print(f"\nFalse positives: {len(false_positive_files)} files")
    if false_positive_files:
        print(f"False positive files:")
        for f in sorted(false_positive_files)[:10]:  # Show first 10
            print(f"  - {f}")
        if len(false_positive_files) > 10:
            print(f"  ... and {len(false_positive_files) - 10} more")
    
    print(f"\nFile-level metrics:")
    print(f"  Precision: {file_precision:.2%}")
    print(f"  Recall: {file_recall:.2%}")
    print(f"  F1-Score: {file_f1:.4f}")
else:
    print("\nNo ground truth available (production mode)")



VALIDATION AGAINST GROUND TRUTH

Ground truth found - validating predictions...

Event-level metrics:
  Precision: 0.00%
  Recall: 0.00%
  F1-Score: 0.0000

Confusion Matrix:
  True Positives (TP): 0
  False Positives (FP): 24
  True Negatives (TN): 7,396
  False Negatives (FN): 0

--------------------------------------------------------------------------------
FILE-LEVEL VALIDATION
--------------------------------------------------------------------------------

Ground truth: 0 timestomped files
Model flagged: 24 files

Detection results:
  Detected: 0/0 (0.0%)
  Missed: 0/0

False positives: 24 files
False positive files:
  - AIRPORT INFORMATION.docx
  - Appraiser_Data.ini
  - Appraiser_TelemetryRunList.xml
  - BladeofGrass.jpg
  - Box Sync.lnk
  - CubaDearmed.jpg
  - DarkWolf.png
  - DeathToll.jpg
  - DemLogic.jpg
  - Dropbox.lnk
  ... and 14 more

File-level metrics:
  Precision: 0.00%
  Recall: 0.00%
  F1-Score: 0.0000


## Cell 8: Feature Importance Analysis


In [459]:
# Cell 8: Feature Importance Analysis

print("\n" + "=" * 80)
print("FEATURE IMPORTANCE ANALYSIS")
print("=" * 80)

# Get feature importance from model
if hasattr(model, 'feature_importances_'):
    importance = model.feature_importances_
    
    # Create importance dataframe using feature_cols from Cell 4
    importance_df = pd.DataFrame({
        'feature': feature_cols,
        'importance': importance
    })
    
    # Sort by importance
    importance_df = importance_df.sort_values('importance', ascending=False)
    
    print(f"\nTop 15 most important features:")
    print(importance_df.head(15).to_string(index=False))
    
    # Validate critical features are important
    print(f"\nValidating critical hybrid features:")
    critical_features = [
        'zero_in_nanoseconds', 'time_reversal_event', 
        'cross_artifact_validation_score', 'basic_info_changed',
        'creation_time_changed_to_past', 'modified_time_changed_to_past',
        'event_count', 'events_in_1min_window'
    ]
    
    for feat in critical_features:
        if feat in importance_df['feature'].values:
            rank = importance_df[importance_df['feature'] == feat].index[0] + 1
            imp = importance_df[importance_df['feature'] == feat]['importance'].values[0]
            print(f"  {feat:40s}: Rank #{rank:2d}, Importance: {imp:.4f}")
        else:
            print(f"  {feat:40s}: Not found in features")
else:
    print("\nFeature importance not available for this model type")



FEATURE IMPORTANCE ANALYSIS

Top 15 most important features:
                        feature  importance
cross_artifact_validation_score    0.863069
                     path_depth    0.042629
             basic_info_changed    0.027369
                       is_image    0.015250
              mft_time_modified    0.012851
        cross_artifact_detected    0.007973
 zero_in_nanoseconds_suspicious    0.004426
            time_reversal_event    0.004258
        lf_modified_time_before    0.003343
                  is_executable    0.003075
                    event_count    0.002287
               in_program_files    0.002169
         zero_in_nanoseconds_lf    0.001893
            in_system_directory    0.001815
         creation_time_modified    0.001614

Validating critical hybrid features:
  zero_in_nanoseconds                     : Rank # 3, Importance: 0.0000
  time_reversal_event                     : Rank # 4, Importance: 0.0043
  cross_artifact_validation_score         : Rank #

## Cell 9: Save Output Files


In [460]:
# Cell 9: Save Output Files

print("\n" + "=" * 80)
print("SAVING OUTPUT FILES")
print("=" * 80)

# 1. Save predictions (all records)
predictions_cols = ['filename', 'full_path', 'lf_lsn', 'usn_usn', 
                   'confidence_pct', 'prediction']
predictions_cols_available = [col for col in predictions_cols if col in df.columns]
predictions_df = df[predictions_cols_available].copy()
predictions_df.to_csv(output_predictions, index=False)

print(f"\n1. Predictions saved:")
print(f"   File: {output_predictions}")
print(f"   Records: {len(predictions_df):,}")
print(f"   Size: {os.path.getsize(output_predictions) / 1024:.2f} KB")

# 2. Save predictions with features (for analysis)
predictions_features_df = df.copy()
predictions_features_df.to_csv(output_predictions_features, index=False)
print(f"\n2. Predictions with features saved:")
print(f"   File: {output_predictions_features}")
print(f"   Records: {len(predictions_features_df):,}")
print(f"   Size: {os.path.getsize(output_predictions_features) / 1024 / 1024:.2f} MB")

# 3. Save flagged files only
flagged_df = df[df['prediction'] == 1].copy()
if len(flagged_df) > 0:
    # Sort by confidence (highest first)
    flagged_df = flagged_df.sort_values('confidence_pct', ascending=False)
    
    # Select columns for flagged files report
    flagged_output_cols = ['filename', 'full_path', 'lf_lsn', 'usn_usn', 'confidence_pct']
    
    # Add key feature columns
    key_features = ['zero_in_nanoseconds', 'zero_in_nanoseconds_lf',
                   'time_reversal_event', 'basic_info_changed', 
                   'cross_artifact_validation_score', 'cross_artifact_detected',
                   'has_logfile_evidence', 'has_usnjrnl_evidence']
    
    for feat in key_features:
        if feat in flagged_df.columns:
            flagged_output_cols.append(feat)
    
    flagged_output_cols_available = [col for col in flagged_output_cols if col in flagged_df.columns]
    flagged_output_df = flagged_df[flagged_output_cols_available].copy()
    flagged_output_df.to_csv(output_flagged, index=False)
    
    print(f"\n3. Flagged files saved:")
    print(f"   File: {output_flagged}")
    print(f"   Records: {len(flagged_output_df):,}")
    print(f"   Size: {os.path.getsize(output_flagged) / 1024:.2f} KB")
    
    # Display first 5 flagged files
    print(f"\n   Top 5 flagged files (by confidence):")
    display_cols = ['filename', 'confidence_pct', 'zero_in_nanoseconds']
    display_cols_available = [col for col in display_cols if col in flagged_output_df.columns]
    print(flagged_output_df[display_cols_available].head().to_string(index=False))
else:
    print(f"\n3. No flagged files to save")

print("\n" + "=" * 80)
print("RUN DETECTION COMPLETE")
print("=" * 80)
print(f"\nSummary:")
print(f"  Total events analyzed: {len(df):,}")
print(f"  Flagged events: {(df['prediction'] == 1).sum():,}")
print(f"  Unique files: {df['filename'].nunique():,}")
print(f"  Flagged files: {len(flagged_df['filename'].unique()) if len(flagged_df) > 0 else 0:,}")
print(f"\nTo test on another dataset:")
print(f"  1. Edit Cell 2 INPUT_DIR to point to another dataset")
print(f"  2. Run all cells")



SAVING OUTPUT FILES



1. Predictions saved:
   File: /Users/soni/Github/Digital-Detectives_Thesis/data/processed/Prototype Tool Output/LW/XGBoostHybrid/predictions.csv
   Records: 7,420
   Size: 857.95 KB

2. Predictions with features saved:
   File: /Users/soni/Github/Digital-Detectives_Thesis/data/processed/Prototype Tool Output/LW/XGBoostHybrid/predictions_with_features.csv
   Records: 7,420
   Size: 3.58 MB

3. Flagged files saved:
   File: /Users/soni/Github/Digital-Detectives_Thesis/data/processed/Prototype Tool Output/LW/XGBoostHybrid/flagged_files.csv
   Records: 24
   Size: 2.83 KB

   Top 5 flagged files (by confidence):
                filename  confidence_pct  zero_in_nanoseconds
AIRPORT INFORMATION.docx       99.951973                    1
             Dropbox.lnk       99.951973                    1
            Box Sync.lnk       99.951973                    1
           Planning.docx       99.951973                    1
        Google Drive.lnk       99.951973                    1

RUN DETEC